In [27]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [29]:
user_id = "user_001"
application_context = "personal_assistant"

namespace = (user_id, application_context)

In [30]:
store.put(
    namespace,
    "memory_001",
    {
        "facts": [
            "사용자는 아아(아이스 아메리카노)를 선호함",
            "사용자는 매일 아침 7시에 일어남"
        ],
        "language": "Korean"
    }
)

In [31]:
item = store.get(namespace, "memory_001")

In [32]:
item

Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아(아이스 아메리카노)를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-02-08T06:13:25.552025+00:00', updated_at='2026-02-08T06:13:25.552025+00:00')

In [33]:
item.value

{'facts': ['사용자는 아아(아이스 아메리카노)를 선호함', '사용자는 매일 아침 7시에 일어남'],
 'language': 'Korean'}

In [34]:
items = store.search(namespace)

In [35]:
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아(아이스 아메리카노)를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-02-08T06:13:25.552025+00:00', updated_at='2026-02-08T06:13:25.552025+00:00', score=None)]

In [36]:
store.put(
    namespace,
    "memory_002",
    {
        "facts": [
            "사용자는 랭체인 공부를 좋아함",
            "학습용 챗봇 프로젝트를 진행하고 있음"
        ],
    }
)

In [37]:
item = store.get(namespace, "memory_002")
item

Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 랭체인 공부를 좋아함', '학습용 챗봇 프로젝트를 진행하고 있음']}, created_at='2026-02-08T06:13:25.673886+00:00', updated_at='2026-02-08T06:13:25.673886+00:00')

In [38]:
items = store.search(namespace)
items

[Item(namespace=['user_001', 'personal_assistant'], key='memory_001', value={'facts': ['사용자는 아아(아이스 아메리카노)를 선호함', '사용자는 매일 아침 7시에 일어남'], 'language': 'Korean'}, created_at='2026-02-08T06:13:25.552025+00:00', updated_at='2026-02-08T06:13:25.552025+00:00', score=None),
 Item(namespace=['user_001', 'personal_assistant'], key='memory_002', value={'facts': ['사용자는 랭체인 공부를 좋아함', '학습용 챗봇 프로젝트를 진행하고 있음']}, created_at='2026-02-08T06:13:25.673886+00:00', updated_at='2026-02-08T06:13:25.673886+00:00', score=None)]

In [39]:
from dataclasses import dataclass

@dataclass
class Context:
    user_id: str
    app_name: str

In [49]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def inject_memory(request, handler):

    current_user = request.runtime.context.user_id
    current_app = request.runtime.context.app_name

    memories = request.runtime.store.search((current_user, current_app))

    memory_content="기록된 정보 없음"
    
    if memories:
        extracted_facts = []
        for item in memories:
            if "facts" in item.value:
                extracted_facts.extend(item.value["facts"])
        memory_content = "\n-".join(extracted_facts)
    
    system_message = f"사용자 관련 장기 메모리 : {memory_content}"
    request = request.override(system_prompt=system_message)
    return handler(request)

In [50]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    middleware=[inject_memory],
    context_schema=Context,
    store=store
)

In [51]:
agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='f9dc87f0-6a89-460c-97a3-65babb461989'),
  AIMessage(content='네, 제가 사용자님에 대해 알고 있는 정보는 다음과 같습니다:\n\n*   **아이스 아메리카노(아아)를 선호합니다.**\n*   **매일 아침 7시에 일어납니다.**\n*   **랭체인 공부를 좋아합니다.**\n*   **현재 학습용 챗봇 프로젝트를 진행하고 있습니다.**', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c3be6-33e9-7bc2-94b6-fbc9d8d79d36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 74, 'output_tokens': 511, 'total_tokens': 585, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 434}})]}